## Imports and connection

In [3]:
import pandas as pd
import sqlite3

con = sqlite3.connect("../data/checking-logs.sqlite")

## Create datamart table with one join query


In [4]:
query = """
CREATE TABLE IF NOT EXISTS datamart AS
SELECT
    c.uid AS uid,
    c.labname AS labname,
    MIN(c.timestamp) AS first_commit_ts,
    MIN(p.datetime) AS first_view_ts
FROM checker c
LEFT JOIN pageviews p
    ON p.uid = c.uid
WHERE c.status = 'ready'
  AND c.numTrials = 1
  AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
  AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname;
"""
con.execute(query)
con.commit()

## Load datamart and parse datetimes

In [3]:
datamart = pd.io.sql.read_sql(
    "SELECT * FROM datamart;",
    con,
    parse_dates=["first_commit_ts", "first_view_ts"]
)
datamart.dtypes
datamart.head(10)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## Create test and control tables

In [4]:
test = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()

avg_first_view = test.drop_duplicates("uid")["first_view_ts"].mean()
control["first_view_ts"] = avg_first_view

test.head()
control.head()

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-26 11:42:34.626102272
13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-26 11:42:34.626102272
14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-26 11:42:34.626102272
15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-26 11:42:34.626102272
16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-26 11:42:34.626102272


## Save test and control back to the database

In [5]:
test.to_sql("test", con, if_exists="replace", index=False)
control.to_sql("control", con, if_exists="replace", index=False)

81

## Close connection